In [56]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix

data = load_diabetes()
X = data.data
y = data.target


چون تارگت این دیتاست پیوسته است، آن را به 3 کلاس تقسیم می‌کنیم

In [ ]:
y_discrete = np.digitize(y, bins=np.percentile(y, [33, 66]))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_clusters = len(np.unique(y_discrete))

print("Number of clusters:", n_clusters)

def clustering_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    print(row_ind, col_ind)
    return cm[row_ind, col_ind].sum() / np.sum(cm)


Number of clusters: 3


# K-Means Clustering

In [58]:
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_acc = clustering_accuracy(y_discrete, kmeans_labels)
kmeans_ari = adjusted_rand_score(y_discrete, kmeans_labels)
kmeans_nmi = normalized_mutual_info_score(y_discrete, kmeans_labels)

print("K-Means Results:")
print("Accuracy:", kmeans_acc)
print("ARI:", kmeans_ari)
print("NMI:", kmeans_nmi)


K-Means Results:
Accuracy: 0.4819004524886878
ARI: 0.09025918507493495
NMI: 0.09612893331404515


# Hierarchical Clustering

In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=n_clusters)
hier_labels = hierarchical.fit_predict(X_scaled)

hier_acc = clustering_accuracy(y_discrete, hier_labels)
hier_ari = adjusted_rand_score(y_discrete, hier_labels)
hier_nmi = normalized_mutual_info_score(y_discrete, hier_labels)

print("Hierarchical Clustering Results:")
print("Accuracy:", hier_acc)
print("ARI:", hier_ari)
print("NMI:", hier_nmi)


Hierarchical Clustering Results:
Accuracy: 0.4796380090497738
ARI: 0.11517363882250217
NMI: 0.10929059343287385


In [60]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X_train, X_test, y_train, y_test = train_test_split(X_scaled,hier_labels,test_size=0.3,random_state=42)

mlp = MLPClassifier(
    hidden_layer_sizes=(10,10),
    max_iter=500,
)

mlp.fit(X_train,y_train)

mlp_score = mlp.score(X_test,y_test)

print("MLPClassifier Results:")
print("Accuracy:", mlp_score)

MLPClassifier Results:
Accuracy: 0.924812030075188


c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [61]:
y_pred = mlp.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print(cm)


[[48  4  2]
 [ 2 45  0]
 [ 0  2 30]]
